In [ ]:
subject_id =170 # or whatever subject you're working with
# set by the runner (papermill); None means: find settings.toml from cwd
config_path = None
# set by the runner, e.g. a test folder for derivatives_root
path_overrides = None


In [ ]:
import mne
import numpy as np
from spectral.epochs import (create_epochs, get_reject_log,
                             describe_artifact_epochs, reject_epochs_fixed)
from spectral.helpers import load_or_create_report, save_subject_report
from spectral.utils import ProjectPaths, print_timestamp, load_config

# raw_annotated = raw_pyprep.copy()
paths = ProjectPaths(subject_id, config_path=config_path, path_overrides=path_overrides)
# One report per subject: 01 creates it, each step adds its own section.
REPORT_SECTION = "02 Epochs"
report = load_or_create_report(paths, subject_id)
fname = paths.preprocessed / f"sub-{subject_id}_annotated_filtered_raw.fif"

if not fname.exists():
    raise FileNotFoundError(f"Could not find preprocessed file: {fname}")

raw_annotated = mne.io.read_raw_fif(fname, preload=True)

In [ ]:
total_duration = raw_annotated.times[-1]

raw_annotated.compute_psd(fmin=1, fmax=50, picks="eeg", exclude="bads").plot(spatial_colors=True)

In [ ]:


# All epoching/rejection parameters come from settings.toml ([epochs]).
_ep = paths.config["epochs"]
epochs = create_epochs(raw_annotated,
                       length=_ep["epoch_length"],
                       overlap=_ep["epoch_overlap"],
                       reject_by_annotation=False)

# create_epochs drops epochs that overlap a recording pause (BAD_ACQ_SKIP),
# also with reject_by_annotation=False (eeg-spectral 0.2.2 and later).
# Recordings with a pause: sub-114, 124, 127, 128, 139.
print(f"Dropped {sum(log == ('BAD_ACQ_SKIP',) for log in epochs.drop_log)} epochs overlapping a recording pause")

# Epoch rejection with autoreject. On a few recordings its threshold search
# fails and returns a threshold near 0 uV, which would reject almost all
# epochs. get_reject_log raises an error in that case; the recording is then
# checked automatically and a fixed peak-to-peak threshold is used instead
# ([epochs] fallback_ptp_uv).
try:
    reject_log = get_reject_log(epochs,
                                resample=_ep["autoreject_resample"],
                                n_interpolate=_ep["autoreject_n_interpolate"],
                                consensus=_ep["autoreject_consensus"])
    bad_epochs = reject_log.bad_epochs
    rejection_method = "autoreject"
except RuntimeError as err:
    reject_log = None
    print(f"!! {err}\n")
    triage = describe_artifact_epochs(epochs)
    print("Automated triage of this recording:")
    for k, v in triage["summary"].items():
        print(f"    {k}: {v}")
    for r in triage["epochs"][:5]:
        print(f"    epoch {r['epoch']:3d} @ {r['onset_s']:7.1f}s  "
              f"{r['n_channels_over']:3d} ch ({r['frac_channels_over']:.0%})  "
              f"max {r['max_ptp_uv']:6d} uV  worst {r['worst_channel']}")
    if not triage["summary"]["remainder_looks_normal"]:
        raise RuntimeError(
            f"sub-{subject_id}: autoreject degenerated AND the remainder does not "
            "look usable. Inspect this recording before including it."
        ) from err
    bad_epochs, fb = reject_epochs_fixed(epochs,
                                         ptp_uv=_ep["fallback_ptp_uv"],
                                         max_bad_frac=_ep["fallback_max_bad_frac"])
    rejection_method = f"fixed: {fb['criterion']}"
    print(f"\nfallback applied: {fb['n_rejected']}/{fb['n_epochs']} epochs "
          f"rejected ({fb['pct_rejected']}%)")

print(f"\nrejection method for sub-{subject_id}: {rejection_method}")
if reject_log is not None:
    reject_plot = reject_log.plot("vertical")


# Drop channels with more than 40% bad epochs
BADS_THRESHOLD = _ep["bad_channel_epoch_threshold"]

if reject_log is not None:
    bads = np.logical_or(reject_log.labels == 1, reject_log.labels == 2)
    bad_channel = bads.mean(axis=0)
    channel_names = epochs.info["ch_names"]
    bad_channels_by_epochs = [
        name for name, value in zip(channel_names, bad_channel) if value > BADS_THRESHOLD
    ]
    print(
        f"Additional {len(bad_channels_by_epochs)} bad channels found:{bad_channels_by_epochs}"
    )
else:
    bad_channels_by_epochs = []
    print("Skipping per-epoch bad-channel analysis (no autoreject labels available).")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Determine if we have any bad epochs
n_bad_epochs = np.sum(bad_epochs)
n_good_epochs = np.sum(~bad_epochs)

# Create the figure
fig, axs = plt.subplots(2, 1, figsize=(10, 10))

# 2. Plot Bad Epochs (with Safeguard)
if n_bad_epochs > 0:
    epochs[bad_epochs].compute_psd(
        method="welch", picks="eeg", fmax=50.0, exclude="bads"
    ).plot(
        average=False, picks="eeg", exclude="bads", show=False,
        amplitude=False, axes=axs[0]
    )
    axs[0].set_title(f"PSD for Bad Epochs (n={n_bad_epochs})")
else:
    # Handle the case where no epochs were dropped
    axs[0].text(0.5, 0.5, 'No Bad Epochs Dropped', 
                ha='center', va='center', fontsize=14)
    axs[0].set_title("PSD for Bad Epochs (None)")
    # Set dummy limits so the ylim comparison doesn't fail
    axs[0].set_ylim(0, 1) 

# 3. Plot Good Epochs (with Safeguard)
if n_good_epochs > 0:
    epochs[~bad_epochs].compute_psd(
        method="welch", picks="eeg", fmax=50.0, exclude="bads"
    ).plot(
        average=False, picks="eeg", exclude="bads", show=False,
        amplitude=False, axes=axs[1]
    )
    axs[1].set_title(f"PSD for Good Epochs (n={n_good_epochs})")
else:
    axs[1].text(0.5, 0.5, 'No Good Epochs Remaining', 
                ha='center', va='center', fontsize=14)

# 4. Synchronize Y-Axis Limits (only if both plots have data)
if n_bad_epochs > 0 and n_good_epochs > 0:
    ylim1 = axs[0].get_ylim()
    ylim2 = axs[1].get_ylim()
    new_ylim = (min(ylim1[0], ylim2[0]), max(ylim1[1], ylim2[1]))
    axs[0].set_ylim(new_ylim)
    axs[1].set_ylim(new_ylim)

plt.tight_layout()
plt.show()

# 5. Add to Report
report.add_figure(fig, title="PSD Comparison: Bad vs Good Epochs",
                  section=REPORT_SECTION, replace=True)

In [ ]:
# Assuming 'raw' is your MNE raw object
# Get the list of all channels
all_channels = epochs.info["ch_names"]

# Get the list of bad channels
bad_channels = epochs.info["bads"]

# Get the list of good channels
good_channels = [ch for ch in all_channels if ch not in bad_channels]

# Get the indices of the good channels
good_channels_indices = [all_channels.index(ch) for ch in good_channels]

# Define the chunk size
chunk_size = 30

# Create chunks of the good channels
chunks = [
    good_channels_indices[i : i + chunk_size]
    for i in range(0, len(good_channels_indices), chunk_size)
]

if n_bad_epochs > 0 :
    # Now you can iterate over the chunks
    for chunk in chunks:
        # Do something with the chunk
        # print(chunk)
        mne.viz.plot_epochs(
            epochs=epochs[bad_epochs],
            picks=chunk,
            show=False,
            scalings=dict(eeg=1e-4),
            n_channels=chunk_size,
            show_scrollbars=False,
        )

In [ ]:
print(f"Sub{subject_id} -  removed {sum(bad_epochs)} epochs")
print(f"Sub{subject_id} -  retained {len(epochs[~bad_epochs])} epochs")
print(f"Sub{subject_id} -  rejection method: {rejection_method}")

# All subjects go through epoch rejection. Where autoreject fails, the fixed
# threshold above is used and reported.
epochs_good = epochs[~bad_epochs]

epochs_good.save(
    f"{paths.epochs}/sub-{subject_id}_good_epochs-epo.fif", overwrite=True
)


In [ ]:
save_subject_report(report, paths, subject_id)